# 10 — WELFake Preprocessing

**Mục tiêu:** Tải WELFake dataset, xử lý null, gộp title+text, áp dụng pipeline NLP, lưu kết quả.

**Input:** Kaggle dataset `saurabhshahane/fake-news-classification`  
**Output:** `data/processed/preprocessed_welfake_full.csv`

> Dataset được tự động tải và cache bằng `kagglehub`; không cần đặt CSV thủ công trong `data/raw/`.  
> Nhãn sau chuẩn hóa theo ISOT hiện tại: `REAL=0`, `FAKE=1`.

## 1. Import & Setup

In [16]:
import sys
import warnings
from pathlib import Path

import kagglehub
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Thêm src/ vào path để import preprocessing
ROOT = Path('..').resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from preprocessing import preprocess_text

DATASET_HANDLE = 'saurabhshahane/fake-news-classification'
DATASET_FILENAME = 'WELFake_Dataset.csv'
OUT_FILE  = ROOT / 'data' / 'processed' / 'preprocessed_welfake_full.csv'
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
(ROOT / 'reports').mkdir(parents=True, exist_ok=True)

print(f'Dataset: {DATASET_HANDLE}')
print(f'Output : {OUT_FILE}')

Dataset: saurabhshahane/fake-news-classification
Output : D:\PROJECT_GIT\Fake-News-Detection\data\processed\preprocessed_welfake_full.csv


## 2. Load & Kiểm tra Dữ liệu Gốc

In [17]:
print('Đang tải WELFake Dataset từ Kaggle...')
dataset_dir = Path(kagglehub.dataset_download(DATASET_HANDLE))

csv_candidates = list(dataset_dir.rglob(DATASET_FILENAME))
if not csv_candidates:
    available_csv = [p.name for p in dataset_dir.rglob('*.csv')]
    raise FileNotFoundError(
        f'Không tìm thấy {DATASET_FILENAME}. Các CSV hiện có: {available_csv}'
    )


Đang tải WELFake Dataset từ Kaggle...


In [22]:

raw_file = csv_candidates[0]
df = pd.read_csv(raw_file)
df = df.rename(columns={'label': 'source_label'})

print(f'Loaded : {raw_file}')
print(f'Shape  : {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print()
print('Null values per column:')
print(df.isnull().sum())
print()
print('Source label distribution (WELFake gốc):')
print(df['source_label'].value_counts().rename({0: 'REAL (0)', 1: 'FAKE (1)'}))
print()
print('Sample rows:')
df.head(3)

Loaded : C:\Users\Nguyen Xuan\.cache\kagglehub\datasets\saurabhshahane\fake-news-classification\versions\77\WELFake_Dataset.csv
Shape  : (72134, 4)
Columns: ['Unnamed: 0', 'title', 'text', 'source_label']

Null values per column:
Unnamed: 0        0
title           558
text             39
source_label      0
dtype: int64

Source label distribution (WELFake gốc):
source_label
FAKE (1)    37106
REAL (0)    35028
Name: count, dtype: int64

Sample rows:


,Unnamed: 0,title,text,source_label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1


In [23]:
# Xem ví dụ bài có null text
null_text = df[df['text'].isnull()]
print(f'\nSố dòng null text: {len(null_text)}')
if len(null_text) > 0:
    print(null_text[['title','text', 'source_label']].head(3).to_string())


Số dòng null text: 39
                                                                                                          title text  source_label
2457  Après le succès de « Mariés au premier regard » M6 va lancer « Fécondées au premier regard » >> Le Gorafi  NaN             1
3534            Elections US : les premières estimations donnent un candidat en avance sur l’autre >> Le Gorafi  NaN             1
3709               110% des Américains assurent qu’ils continueront à faire confiance aux sondages >> Le Gorafi  NaN             1


## 3. Xử lý Giá trị Thiếu

In [24]:
n_before = len(df)

# Drop dòng null cả title lẫn text
both_null = df['title'].isnull() & df['text'].isnull()
df = df[~both_null].copy()
print(f'Drop cả hai null: {both_null.sum()} dòng → còn {len(df):,}')

# Fill đơn lẻ
df['title'] = df['title'].fillna('')
df['text']  = df['text'].fillna('')

print(f'Tổng drop: {n_before - len(df)} dòng')
print(f'Null còn lại - title: {df["title"].isnull().sum()}, text: {df["text"].isnull().sum()}')

Drop cả hai null: 0 dòng → còn 72,134
Tổng drop: 0 dòng
Null còn lại - title: 0, text: 0


## 4. Gộp Title + Text

In [25]:
# Gộp: title + " " + text
df['full_text'] = df['title'].str.strip() + ' ' + df['text'].str.strip()
df['full_text'] = df['full_text'].str.strip()

print(f'Độ dài full_text (words):')
word_counts = df['full_text'].str.split().str.len()
print(f'  Min: {word_counts.min()}')
print(f'  Max: {word_counts.max()}')
print(f'  Mean: {word_counts.mean():.0f}')
print(f'  Median: {word_counts.median():.0f}')

# Cảnh báo bài quá ngắn (< 10 words)
very_short = (word_counts < 10).sum()
print(f'\nBài < 10 words: {very_short} ({very_short/len(df)*100:.1f}%)')

Độ dài full_text (words):
  Min: 1
  Max: 24243
  Mean: 553
  Median: 410

Bài < 10 words: 220 (0.3%)


## 5. Áp dụng Preprocessing Pipeline

In [26]:
# Demo trên 1 bài trước khi chạy toàn bộ
sample_raw = df['full_text'].iloc[0]
sample_proc = preprocess_text(sample_raw)
print('=== BEFORE ===')
print(sample_raw[:300])
print('\n=== AFTER ===')
print(sample_proc[:300])

=== BEFORE ===
LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO] No comment is expected from Barack Obama Members of the #FYF911 or #FukYoFlag and #BlackLivesMatter movements called for the lynching and hanging of white people and cop

=== AFTER ===
law enforcement high alert following threat cop white blacklivesmatter fyf terrorist video no comment expected barack obama member fyf fukyoflag blacklivesmatter movement called lynching hanging white people cop encouraged others radio show tuesday night turn tide kill white people cop send message 


In [27]:
import time
print(f'Đang xử lý {len(df):,} bài báo...')
t0 = time.time()

# Xử lý theo batch với progress reporting
BATCH = 5000
results = []
for i in range(0, len(df), BATCH):
    batch = df['full_text'].iloc[i:i+BATCH].apply(preprocess_text)
    results.append(batch)
    pct = min(i+BATCH, len(df)) / len(df) * 100
    elapsed = time.time() - t0
    print(f'  {pct:5.1f}%  [{i+len(batch):,}/{len(df):,}]  {elapsed:.0f}s')

df['processed_text'] = pd.concat(results)
total_time = time.time() - t0
print(f'\nHoàn thành trong {total_time:.0f}s ({total_time/len(df)*1000:.1f}ms/bài)')

Đang xử lý 72,134 bài báo...
    6.9%  [5,000/72,134]  16s
   13.9%  [10,000/72,134]  32s
   20.8%  [15,000/72,134]  55s
   27.7%  [20,000/72,134]  77s
   34.7%  [25,000/72,134]  99s
   41.6%  [30,000/72,134]  121s
   48.5%  [35,000/72,134]  146s
   55.5%  [40,000/72,134]  167s
   62.4%  [45,000/72,134]  188s
   69.3%  [50,000/72,134]  209s
   76.2%  [55,000/72,134]  231s
   83.2%  [60,000/72,134]  248s
   90.1%  [65,000/72,134]  264s
   97.0%  [70,000/72,134]  279s
  100.0%  [72,134/72,134]  286s

Hoàn thành trong 286s (4.0ms/bài)


## 6. Kiểm tra Kết quả

In [29]:
# Kiểm tra empty strings sau preprocessing
empty_after = (df['processed_text'].str.strip() == '').sum()
print(f'Bài trống sau preprocessing: {empty_after}')

if empty_after > 0:
    print('Drop bài trống...')
    df = df[df['processed_text'].str.strip() != ''].copy()
    print(f'Còn lại: {len(df):,}')

# Phân phối cuối
print(f'\nPhân phối nhãn cuối cùng:')
vc = df['source_label'].value_counts().sort_index()
for label, count in vc.items():
    name = 'REAL' if label == 0 else 'FAKE'
    print(f'  {name} ({label}): {count:,}  ({count/len(df)*100:.1f}%)')

print(f'\nTổng số mẫu: {len(df):,}')

# So sánh với ISOT
print('\n=== So sánh với ISOT ===')
print(f'  ISOT  : ~44,898 mẫu')
print(f'  WELFake: {len(df):,} mẫu  ({len(df)/44898*100:.0f}% so với ISOT)')

Bài trống sau preprocessing: 0

Phân phối nhãn cuối cùng:
  REAL (0): 35,028  (48.6%)
  FAKE (1): 37,046  (51.4%)

Tổng số mẫu: 72,074

=== So sánh với ISOT ===
  ISOT  : ~44,898 mẫu
  WELFake: 72,074 mẫu  (161% so với ISOT)


In [30]:
# Biểu đồ độ dài text trước/sau preprocessing
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

before_len = df['full_text'].str.split().str.len()
after_len  = df['processed_text'].str.split().str.len()

for ax, data, title in zip(axes,
    [before_len, after_len],
    ['Trước preprocessing (words)', 'Sau preprocessing (tokens)']):
    ax.hist(data.clip(upper=data.quantile(0.99)), bins=50, color='steelblue', alpha=0.7, edgecolor='white')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Số từ', fontsize=10)
    ax.set_ylabel('Số bài', fontsize=10)
    ax.axvline(data.median(), color='red', linestyle='--', alpha=0.7, label=f'Median={data.median():.0f}')
    ax.legend(fontsize=9)

plt.suptitle('WELFake — Phân phối độ dài bài báo', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(str(ROOT / 'reports' / 'welfake_text_length.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/welfake_text_length.png')

Saved → reports/welfake_text_length.png


## 7. Lưu Kết quả

In [32]:

df.rename(columns={'source_label': 'label'}, inplace=True)
output_df = df[['processed_text', 'label']].copy()
output_df['label'] = output_df['label'].astype(int)
output_df.to_csv(OUT_FILE, index=False)

size_mb = OUT_FILE.stat().st_size / 1024 / 1024
print(f'✅ Saved: {OUT_FILE}')
print(f'   Rows : {len(output_df):,}')
print(f'   Size : {size_mb:.1f} MB')
print(f'   Cols : {output_df.columns.tolist()}')

# Verify đọc lại được
verify = pd.read_csv(OUT_FILE)
assert len(verify) == len(output_df), 'Mismatch!'
print(f'\nVerify đọc lại: OK — {len(verify):,} rows ✓')

✅ Saved: D:\PROJECT_GIT\Fake-News-Detection\data\processed\preprocessed_welfake_full.csv
   Rows : 72,074
   Size : 155.1 MB
   Cols : ['processed_text', 'label']

Verify đọc lại: OK — 72,074 rows ✓


## Tóm tắt

| Bước | Kết quả |
|------|---------|
| Tải dữ liệu | KaggleHub cache, ~72,134 rows |
| Xử lý null | Drop dòng null cả hai cột |
| Gộp title+text | `full_text` = title + " " + text |
| Chuẩn hóa nhãn | WELFake gốc → `REAL=0`, `FAKE=1` như ISOT |
| Preprocessing | `preprocess_text()` từ src/ |
| Output | `data/processed/preprocessed_welfake_full.csv` |